In [9]:
import pandas as pd
import glob
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- BƯỚC 1: ĐỌC DỮ LIỆU TỪ 10 FILE ---
path = './VLSP2018-SA-train-dev-test' 
all_files = glob.glob(os.path.join(path, "*.txt"))

texts, labels = [], []

for filename in all_files:
    with open(filename, 'r', encoding='utf-8-sig') as f:
        # Tách từng cụm (cách nhau bởi dòng trống)
        blocks = f.read().strip().split('\n\n')
        for block in blocks:
            lines = block.split('\n')
            if len(lines) >= 3:
                content = lines[1] # Dòng chữ
                sentiment_line = lines[2].lower() # Dòng nhãn
                
                # Gán nhãn: 2 (Pos), 0 (Neg), 1 (Neu)
                if 'negative' in sentiment_line: label = 0
                elif 'positive' in sentiment_line: label = 2
                else: label = 1
                
                texts.append(content)
                labels.append(label)

# --- BƯỚC 2: BIẾN CHỮ THÀNH SỐ (Lý thuyết Chương 4) ---
# token_pattern giúp máy nhận diện được tiếng Việt có dấu
vectorizer = TfidfVectorizer(ngram_range=(1, 2), token_pattern=r'(?u)\b\w\w+\b')
X = vectorizer.fit_transform(texts)
y = labels

# --- BƯỚC 3: HUẤN LUYỆN SOFTMAX REGRESSION ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# multi_class='multinomial' chính là Softmax Regression trong sách
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

# --- BƯỚC 4: KẾT QUẢ ---
print(f"Đã học xong từ {len(texts)} mẫu dữ liệu!")
print("Độ chính xác chi tiết:")
print(classification_report(y_test, model.predict(X_test), target_names=['Neg', 'Neu', 'Pos']))
y_pred=model.predict(X_test)
print(accuracy_score(y_test, y_pred))

C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Đã học xong từ 10351 mẫu dữ liệu!
Độ chính xác chi tiết:
              precision    recall  f1-score   support

         Neg       0.87      0.67      0.76       704
         Neu       0.00      0.00      0.00        38
         Pos       0.83      0.95      0.89      1329

    accuracy                           0.84      2071
   macro avg       0.57      0.54      0.55      2071
weighted avg       0.83      0.84      0.83      2071

0.8396909705456301


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize

In [34]:
def du_doan_cua_ban(cau_moi):
    # 1. Biến câu chữ thành Vector số (phải dùng đúng cái vectorizer đã fit ở trên)
    vector_cau_moi = vectorizer.transform([cau_moi])
    
    # 2. Dự đoán nhãn (0, 1, hoặc 2)
    ket_qua = model.predict(vector_cau_moi)[0]
    
    # 3. Xem xác suất phần trăm cho từng lớp (đúng tinh thần Softmax)
    xac_suat = model.predict_proba(vector_cau_moi)[0]
    
    # Mapping số về chữ cho dễ đọc
    labels_map = {0: "Tiêu cực (Negative)", 1: "Trung tính (Neutral)", 2: "Tích cực (Positive)"}
    
    print(f"\n--- KẾT QUẢ DỰ ĐOÁN ---")
    print(f"Câu của bạn: '{cau_moi}'")
    print(f"Dự đoán: {labels_map[ket_qua]}")
    print(f"Độ tự tin: Neg: {xac_suat[0]:.2f}, Neu: {xac_suat[1]:.2f}, Pos: {xac_suat[2]:.2f}")

# --- TEST THỬ ---
my_sentences = "nhân viên cọc tính, đồ ăn phục vụ khá lâu và nó dở, nhà hàng không sang sọng, tôi sẽ không ăn ở đây thêm lần nữa."
ex2='nói chung là bình thường'
du_doan_cua_ban(ex2)


--- KẾT QUẢ DỰ ĐOÁN ---
Câu của bạn: 'nói chung là bình thường'
Dự đoán: Tích cực (Positive)
Độ tự tin: Neg: 0.21, Neu: 0.34, Pos: 0.45


In [35]:
print(f"Số lượng feature (từ vựng): {len(vectorizer.get_feature_names_out())}")

Số lượng feature (từ vựng): 173916
